# 2.1 Calculate degassing path using VESIcal

<div style="background-color:#eef8fa; border-left:4px solid #24bdff; padding:10px; border-radius:4px;">
<b> 🧮 &nbsp; VESIcal </b> is a framework for thermodynamic modeling of magmatic volatiles written in Python - an open-source volatile solubility engine.

More information on VESIcal can be found at https://vesical.readthedocs.io/en/latest/

</div>

## 2.1.1 Import packages and note versions

In [ ]:
# Packages that we will use in our code always get imported before we need them.
# This is canonically done at the top of a script.
# ⚠️ Note that this can take a few seconds if it's the first time you're importing these libraries.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import VESIcal as vc

import os
os.makedirs("output", exist_ok=True) # creates the output folder

When reporting calculations in manuscripts, it's important to know which version of the Python package the results you are showing used - so we can output those below. This notebook was created using VESIcal: 1.2.12.

In [ ]:
print(f"\nVESIcal: {vc.__version__}")

## 2.1.2 Import data

We'll use the same melt inclusion dataset from notebook <a href="1_1_MI_Temperature_Thermobar.ipynb">1.1 Calculate temperature from MI data using Thermobar</a>, where we've already calculated temperatures.

In [ ]:
# import melt inclusion dataset
MI = pd.read_csv("output/wieser2021_w_temperatures.csv")

# if you haven't run Exercise 1, you can grab the "answer key" file from here - remove the # at the start of the line below and then run the cell
#results_pvsat_vf = pd.read_csv("answers/wieser2021_w_temperatures.csv")

## 2.1.3 Explore options

For consistency with the previous Exercise, we'll use the same model options as before (``"IaconoMarziano"``), but these can be changed! See Section 1.2.3 in <a href="1_2_MI_Pressure_FluidComp_VESIcal.ipynb">1.2 Calculate saturation pressures and equilibrium fluid compositions from MI data using VESIcal</a> or the VESIcal ReadTheDocs for more info.

## 2.1.4 Initial melt composition

We choose the MI with the highest CO<sub>2</sub> content as the starting point for the degassing calcuations

In [ ]:
# row number with highest CO2 content
row = MI['CO2'].idxmax()

print(f"The sample with the highest CO2 content is {row} with {MI['CO2'][row]} wt%")

And combine this all into a single dictionary to use in the calculations in the following sections

In [ ]:
ini_comp = MI.loc[row].to_dict() # convert to dictionary
ini_comp = {k.split(" ")[0]: v for k, v in ini_comp.items()}

# print initial composition
ini_comp

And create a VESIcal sample from this dictionary

In [ ]:
# create VESIcal sample from ini_comp
ini_comp_wtpt = {key: val for key, val in ini_comp.items() if key in vc.oxides}
sample_vc = vc.Sample(ini_comp_wtpt)
# sample_vc.get_composition() # print composition if so desired

## 2.1.5 Run calculation

In [ ]:
# run calculation
# temperature is specified in the function
results_degas_vc = vc.calculate_degassing_path(
    sample=sample_vc,
    temperature=float(ini_comp['T_C']), # temperatures calculated by Thermobar in Exercise 1.1
    model="IaconoMarziano").result

# save your results
results_degas_vc.to_csv("output/results_degas_vc.csv")

# uncomment the line below to interrogate the resulting dataframe
#results_degas_vc

## 2.1.6 Plotting

And we can plot against the pressure and fluid composition <a href="1_2_MI_Pressure_FluidComp_VESIcal.ipynb">1.2 Calculate saturation pressures and equilibrium fluid compositions from MI data using VESIcal</a>

In [ ]:
results_pvsat_vc = pd.read_csv("output/results_pvsat_vc.csv")

# if you haven't run Exercise 1.2, you can grab the "answer key" file from here - remove the # at the start of the line below and then run the cell
#results_pvsat_vc = pd.read_csv("answers/results_pvsat_vc.csv")

In [ ]:
# if you haven't run this Exercise, you can grab the "answer key" files from here - remove the # at the start of the line below and then run the cell
#results_degas_vc = pd.read_csv("answers/results_degas_vc.csv")

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12,4)) # figure with three panels in a row

# melt compositions as circles, fluid compositions as diamonds

# VESIcal MI results in red from Exercise 1
df = results_pvsat_vc
ax1.plot(df['H2O'], df['SaturationP_bars_VESIcal'], 'ok', mfc='red', label = "MI VESIcal_IM")
ax2.plot(df['CO2']*10000, df['SaturationP_bars_VESIcal'], 'ok', mfc='red')
ax3.plot(df['XCO2_fl_VESIcal']*100, df['SaturationP_bars_VESIcal'], 'dk', mfc='red')

# VESIcal degassing results from this exercise in red
df = results_degas_vc
ax1.plot(df['H2O_liq'], df['Pressure_bars'], '-', color='red', linewidth=2, label = "VESIcal_IM")
ax2.plot(df['CO2_liq']*10000, df['Pressure_bars'], '-', color='red', linewidth=2)
ax3.plot(df['CO2_fl']*100, df['Pressure_bars'], '-', color='red', linewidth=2)

# label axes
ax1.set_ylabel('P (bars)')
ax1.set_xlabel('H$_2$O [m] (wt%)')
ax2.set_xlabel('CO$_2$ [m] (ppm)')
ax3.set_xlabel('CO$_2$ [f] (mol%)')

# add legend
ax1.legend(frameon=False)

plt.tight_layout()